# 🇮🇳 Telugu Question Answering - Training Notebook

Train **MuRIL** or **IndicBERT** on the TeQuAD dataset.

## 📦 Before You Start: What to Upload

Run this command locally to create the training zip:
```bash
python scripts/export_for_colab.py
```

This creates `colab-upload.zip` containing:
```
colab-upload.zip
├── data/
│   └── processed/
│       ├── tequad_train.json      # Training data (SQuAD format)
│       └── tequad_validation.json # Validation data
└── config/
    └── training_config.yaml       # Training parameters
```

## ⏱️ Training Time (Colab T4 GPU)

| Model | Parameters | Time | Expected F1 |
|-------|-----------|------|-------------|
| **MuRIL** | 237M | ~2-3 hours | 79-84% |
| **IndicBERT** | 33M | ~1-2 hours | 35-40% |

## 📥 After Training: Download These Files

From `models/checkpoints/<model>/`:
- `config.json`
- `model.safetensors` (or `pytorch_model.bin`)
- `tokenizer.json`
- `tokenizer_config.json`
- `special_tokens_map.json`

---

## 1️⃣ Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate sentencepiece pyyaml

In [ ]:
# Mount Google Drive for saving checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Create output directory in Drive
import os
DRIVE_OUTPUT = '/content/drive/MyDrive/telugu-qa-checkpoints'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_OUTPUT}")

## 2️⃣ Upload Project Files

Upload your project zip or clone from GitHub.

In [ ]:
# Option A: Upload zip file
from google.colab import files
import zipfile
import os

print("Upload your colab-upload.zip file:")
uploaded = files.upload()

# Extract to /content
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content')
        print(f"Extracted {filename}")

# Fix folder structure - move processed/ into data/
os.makedirs('/content/data', exist_ok=True)
if os.path.exists('/content/processed'):
    !mv /content/processed /content/data/
    print("Moved processed/ → data/processed/")

print("\nExtracted structure:")
!ls -la /content

In [ ]:
# Set project root and add to path
import sys
PROJECT_ROOT = '/content'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Verify structure
print("Project structure:")
!ls -la
print("\nSource modules:")
!ls -la src/
print("\nData files:")
!ls -la data/processed/

## 3️⃣ Load Dataset

In [ ]:
from src.data.tequad_loader import load_tequad_dataset, TeQuADDataset

# Load all splits
dataset = TeQuADDataset()
print(dataset)

# Show sample
print("\nSample from training set:")
sample = dataset.train[0]
print(f"Question: {sample['question']}")
print(f"Answer: {sample['answers']['text'][0]}")
print(f"Context: {sample['context'][:200]}...")

## 4️⃣ Training Configuration

In [ ]:
# Training configuration
CONFIG = {
    # Model selection: "muril" (best) or "indicbert" (faster)
    "model_key": "muril",
    
    # Training params (optimized for Colab T4)
    "learning_rate": 3e-5,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 2,
    
    # Preprocessing
    "max_seq_length": 384,
    "doc_stride": 128,
    
    # Evaluation
    "eval_steps": 1000,
    "save_steps": 1000,
    
    # Output
    "output_dir": DRIVE_OUTPUT,
}

print("Training Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 5️⃣ Load Model

In [ ]:
from src.models.model_factory import load_model_and_tokenizer, ModelFactory

# Show available models
ModelFactory.list_models()

# Load selected model
model, tokenizer = load_model_and_tokenizer(CONFIG["model_key"])

## 6️⃣ Preprocess Data

In [ ]:
from src.data.preprocessing import preprocess_for_training, preprocess_validation

print("Preprocessing training data...")
train_dataset = preprocess_for_training(
    dataset.train,
    tokenizer,
    max_length=CONFIG["max_seq_length"],
    doc_stride=CONFIG["doc_stride"],
    num_proc=1
)
print(f"Training examples: {len(train_dataset)}")

print("\nPreprocessing validation data...")
eval_dataset = preprocess_validation(
    dataset.validation,
    tokenizer,
    max_length=CONFIG["max_seq_length"],
    doc_stride=CONFIG["doc_stride"],
    num_proc=1
)
print(f"Validation examples: {len(eval_dataset)}")

## 7️⃣ Setup Trainer

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator
import torch

# Create output directory for this model
model_output_dir = os.path.join(CONFIG["output_dir"], CONFIG["model_key"])
os.makedirs(model_output_dir, exist_ok=True)

# Calculate warmup steps (10% of total steps)
total_steps = (len(train_dataset) // (CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"])) * CONFIG["num_train_epochs"]
warmup_steps = int(0.1 * total_steps)

training_args = TrainingArguments(
    output_dir=model_output_dir,
    
    # Training
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    
    # Optimization
    fp16=True,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    
    # Evaluation & Saving
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    
    # Logging
    logging_steps=100,
    logging_first_step=True,
    report_to="none",
)

# For QA, we need to preprocess eval_dataset the same way as train_dataset
# to get proper loss computation
eval_dataset_for_loss = preprocess_for_training(
    dataset.validation,
    tokenizer,
    max_length=CONFIG["max_seq_length"],
    doc_stride=CONFIG["doc_stride"],
    num_proc=1
)
print(f"Eval examples for loss tracking: {len(eval_dataset_for_loss)}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset_for_loss,  # Now includes start/end positions for loss
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

print(f"\n✓ Trainer ready!")
print(f"  Output: {model_output_dir}")
print(f"  Epochs: {CONFIG['num_train_epochs']}")
print(f"  Total steps: {total_steps}, Warmup: {warmup_steps}")
print(f"  Batch size: {CONFIG['per_device_train_batch_size']} x {CONFIG['gradient_accumulation_steps']} = {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"  Evaluation every {CONFIG['eval_steps']} steps")
print(f"  Checkpoints saved every {CONFIG['save_steps']} steps")
print(f"\n📊 Watch for: eval_loss decreasing → good | eval_loss increasing → overfitting")

## 8️⃣ Train Model 🚀

In [ ]:
# Start training!
print("="*60)
print(f"🚀 Starting training: {CONFIG['model_key'].upper()}")
print("="*60)

# Optionally resume from checkpoint
resume_checkpoint = None  # Set to checkpoint path to resume

train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

print("\n" + "="*60)
print("✅ Training Complete!")
print("="*60)
print(f"\nMetrics:")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

## 9️⃣ Save Final Model

In [ ]:
# Save the final model
final_model_path = os.path.join(model_output_dir, "final")
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"\n✓ Model saved to: {final_model_path}")

# List saved files
print("\nSaved files:")
for f in os.listdir(final_model_path):
    size = os.path.getsize(os.path.join(final_model_path, f))
    print(f"  {f}: {size/1024/1024:.1f} MB")

## 🔟 Test the Model

In [ ]:
from transformers import pipeline

# Load the trained model
qa_pipeline = pipeline(
    "question-answering",
    model=final_model_path,
    tokenizer=final_model_path,
    device=0  # GPU
)

# Test with Telugu example
context = "హైదరాబాద్ తెలంగాణ రాష్ట్ర రాజధాని. ఇది దక్కన్ పీఠభూమిపై ఉంది. హైదరాబాద్ జనాభా దాదాపు 1 కోటి."
question = "తెలంగాణ రాజధాని ఏది?"

result = qa_pipeline(question=question, context=context)

print("Test Result:")
print(f"  Context: {context}")
print(f"  Question: {question}")
print(f"  Answer: {result['answer']}")
print(f"  Score: {result['score']:.4f}")

## 📥 Download Model

Download the trained model to your local machine.

In [ ]:
# Create a zip of the model for download
import shutil

zip_name = f"telugu-qa-{CONFIG['model_key']}-trained"
shutil.make_archive(zip_name, 'zip', final_model_path)

print(f"\nModel zipped: {zip_name}.zip")
print("\nDownload from Google Drive or use:")

# Optional: Direct download
# from google.colab import files
# files.download(f"{zip_name}.zip")

---

## 🔄 Train Second Model (Optional)

If you have time remaining, train IndicBERT as a lightweight alternative.

In [ ]:
# Uncomment to train IndicBERT

# # Clear GPU memory
# import torch
# del model, trainer
# torch.cuda.empty_cache()

# # Change config
# CONFIG["model_key"] = "indicbert"

# # Then run cells 5-9 again

---

## 📋 Summary

After training MuRIL, you should have:

1. **Trained model** saved to Google Drive
2. **Checkpoints** for resuming if needed
3. **Test results** showing the model works

**Next Steps:**
1. Download the model to your local machine
2. Copy to `models/checkpoints/muril/`
3. Continue to train IndicBERT below for comparison!

---

# 🔄 Train IndicBERT (Comparison Model)

Now train IndicBERT on the same dataset to compare performance.

**Why Compare Models?**
- MuRIL: Best for Indian languages, 237M parameters
- IndicBERT: Lightweight (33M params), faster inference
- Same training data → fair comparison

**Expected Results:**
- MuRIL: ~73% EM, ~84% F1
- IndicBERT: ~65-70% EM, ~78-82% F1 (lighter model)

## I1️⃣ IndicBERT Configuration

In [ ]:
# Clear GPU memory from MuRIL training
import torch
if 'model' in dir():
    del model
if 'trainer' in dir():
    del trainer
torch.cuda.empty_cache()

# IndicBERT uses a different output directory
INDICBERT_DRIVE_OUTPUT = '/content/drive/MyDrive/telugu-qa-BERT-checkpoints'
os.makedirs(INDICBERT_DRIVE_OUTPUT, exist_ok=True)

# IndicBERT Configuration (same training params as MuRIL)
INDICBERT_CONFIG = {
    "model_key": "indicbert",
    
    # Same training params for fair comparison
    "learning_rate": 3e-5,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 16,  # Can use larger batch (smaller model)
    "per_device_eval_batch_size": 32,
    "gradient_accumulation_steps": 1,
    
    # Preprocessing
    "max_seq_length": 384,
    "doc_stride": 128,
    
    # Evaluation
    "eval_steps": 1000,
    "save_steps": 1000,
    
    # Output (different from MuRIL)
    "output_dir": INDICBERT_DRIVE_OUTPUT,
}

print("⚙️ IndicBERT Training Configuration:")
print("="*50)
for k, v in INDICBERT_CONFIG.items():
    print(f"  {k}: {v}")

print(f"\n📊 Comparison: MuRIL (237M) vs IndicBERT (33M)")
print(f"\n📁 Paths:")
print(f"   MuRIL:     {DRIVE_OUTPUT}/muril")
print(f"   IndicBERT: {INDICBERT_DRIVE_OUTPUT}/indicbert")

## I2️⃣ Load IndicBERT Model

In [ ]:
from src.models.model_factory import load_model_and_tokenizer

# Load IndicBERT
indicbert_model, indicbert_tokenizer = load_model_and_tokenizer("indicbert")

print(f"✓ Model: {indicbert_model.config._name_or_path}")
print(f"  Parameters: {sum(p.numel() for p in indicbert_model.parameters()):,}")

## I3️⃣ Preprocess Data for IndicBERT

In [ ]:
from src.data.preprocessing import preprocess_for_training, preprocess_validation

print("Preprocessing training data for IndicBERT...")
indicbert_train = preprocess_for_training(
    dataset.train,
    indicbert_tokenizer,
    max_length=INDICBERT_CONFIG["max_seq_length"],
    doc_stride=INDICBERT_CONFIG["doc_stride"],
    num_proc=1
)
print(f"Training examples: {len(indicbert_train)}")

print("\nPreprocessing validation data...")
indicbert_eval = preprocess_for_training(
    dataset.validation,
    indicbert_tokenizer,
    max_length=INDICBERT_CONFIG["max_seq_length"],
    doc_stride=INDICBERT_CONFIG["doc_stride"],
    num_proc=1
)
print(f"Validation examples: {len(indicbert_eval)}")

## I4️⃣ Setup IndicBERT Trainer

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator
import os

# Output directory for IndicBERT
indicbert_output = os.path.join(INDICBERT_CONFIG["output_dir"], "indicbert")
os.makedirs(indicbert_output, exist_ok=True)

# Calculate warmup steps
total_steps = (len(indicbert_train) // (INDICBERT_CONFIG["per_device_train_batch_size"] * INDICBERT_CONFIG["gradient_accumulation_steps"])) * INDICBERT_CONFIG["num_train_epochs"]
warmup_steps = int(0.1 * total_steps)

indicbert_args = TrainingArguments(
    output_dir=indicbert_output,
    
    # Training
    learning_rate=INDICBERT_CONFIG["learning_rate"],
    num_train_epochs=INDICBERT_CONFIG["num_train_epochs"],
    per_device_train_batch_size=INDICBERT_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=INDICBERT_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=INDICBERT_CONFIG["gradient_accumulation_steps"],
    
    # Optimization
    fp16=True,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    
    # Evaluation & Saving
    eval_strategy="steps",
    eval_steps=INDICBERT_CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=INDICBERT_CONFIG["save_steps"],
    save_total_limit=3,
    
    # Logging
    logging_steps=100,
    logging_first_step=True,
    report_to="none",
)

indicbert_trainer = Trainer(
    model=indicbert_model,
    args=indicbert_args,
    train_dataset=indicbert_train,
    eval_dataset=indicbert_eval,
    processing_class=indicbert_tokenizer,
    data_collator=default_data_collator,
)

print(f"\n✓ IndicBERT Trainer ready!")
print(f"  Output: {indicbert_output}")
print(f"  Epochs: {INDICBERT_CONFIG['num_train_epochs']}")
print(f"  Total steps: {total_steps}")
print(f"  Batch size: {INDICBERT_CONFIG['per_device_train_batch_size']}")

## I5️⃣ Train IndicBERT 🚀

In [ ]:
print("="*60)
print("🚀 Starting IndicBERT Training")
print("="*60)

indicbert_result = indicbert_trainer.train()

print("\n" + "="*60)
print("✅ IndicBERT Training Complete!")
print("="*60)
print(f"\nMetrics:")
for key, value in indicbert_result.metrics.items():
    print(f"  {key}: {value}")

## I6️⃣ Save IndicBERT Model

In [ ]:
# Save the BEST model (checkpoint-11000 has lowest validation loss: 3.34)
import shutil

best_checkpoint = os.path.join(indicbert_output, "checkpoint-11000")
indicbert_final = os.path.join(indicbert_output, "final")

# Copy best checkpoint as final model
if os.path.exists(best_checkpoint):
    shutil.copytree(best_checkpoint, indicbert_final, dirs_exist_ok=True)
    print(f"✓ Copied best checkpoint (11000) to: {indicbert_final}")
else:
    # Fallback: save current trainer state
    indicbert_trainer.save_model(indicbert_final)
    indicbert_tokenizer.save_pretrained(indicbert_final)
    print(f"✓ Saved trainer model to: {indicbert_final}")

# List saved files
print("\nSaved files:")
for f in os.listdir(indicbert_final):
    size = os.path.getsize(os.path.join(indicbert_final, f))
    print(f"  {f}: {size/1024/1024:.1f} MB")

print(f"\n📊 Best checkpoint selected based on validation loss:")
print(f"   checkpoint-11000: val_loss = 3.339998 (lowest)")

## I7️⃣ Compare Both Models

In [ ]:
from transformers import pipeline

print("Loading both models for comparison...")
print("  MuRIL: checkpoint-9000 (best val_loss: 1.855)")
print("  IndicBERT: checkpoint-11000 (best val_loss: 3.340)")

# Paths (different base directories)
MURIL_BASE = '/content/drive/MyDrive/telugu-qa-checkpoints/muril'
INDICBERT_BASE = '/content/drive/MyDrive/telugu-qa-BERT-checkpoints/indicbert'

# MuRIL - best checkpoint is 9000
muril_path = os.path.join(MURIL_BASE, "checkpoint-9000")
muril_qa = pipeline("question-answering", model=muril_path, tokenizer=muril_path, device=0)

# IndicBERT - best checkpoint is 11000 (saved as final)
indicbert_path = os.path.join(INDICBERT_BASE, "final")
indicbert_qa = pipeline("question-answering", model=indicbert_path, tokenizer=indicbert_path, device=0)

# Test cases
test_cases = [
    {
        "context": "హైదరాబాద్ తెలంగాణ రాష్ట్ర రాజధాని. ఇది దక్కన్ పీఠభూమిపై ఉంది.",
        "question": "తెలంగాణ రాజధాని ఏది?"
    },
    {
        "context": "భారతదేశం దక్షిణ ఆసియాలో ఉన్న దేశం. భారతదేశ రాజధాని న్యూఢిల్లీ.",
        "question": "భారతదేశ రాజధాని ఏది?"
    },
    {
        "context": "విజయవాడ ఆంధ్రప్రదేశ్ లోని ముఖ్యమైన నగరం. ఇది కృష్ణా నది ఒడ్డున ఉంది.",
        "question": "విజయవాడ ఏ నది ఒడ్డున ఉంది?"
    }
]

print("\n" + "="*70)
print("📊 MODEL COMPARISON: MuRIL (checkpoint-9000) vs IndicBERT (checkpoint-11000)")
print("="*70)

muril_wins = 0
indicbert_wins = 0

for i, tc in enumerate(test_cases, 1):
    print(f"\nTest {i}: {tc['question']}")
    
    muril_result = muril_qa(question=tc['question'], context=tc['context'])
    indicbert_result = indicbert_qa(question=tc['question'], context=tc['context'])
    
    print(f"  MuRIL:     {muril_result['answer']} ({muril_result['score']:.2%})")
    print(f"  IndicBERT: {indicbert_result['answer']} ({indicbert_result['score']:.2%})")
    
    if muril_result['score'] > indicbert_result['score']:
        muril_wins += 1
        print("  → MuRIL more confident")
    else:
        indicbert_wins += 1
        print("  → IndicBERT more confident")

print("\n" + "="*70)
print(f"📈 Summary: MuRIL won {muril_wins}/{len(test_cases)}, IndicBERT won {indicbert_wins}/{len(test_cases)}")
print("="*70)

## I8️⃣ Download IndicBERT

In [ ]:
import shutil

indicbert_zip = "telugu-qa-indicbert-trained"
shutil.make_archive(indicbert_zip, 'zip', indicbert_final)

print(f"\n✓ IndicBERT zipped: {indicbert_zip}.zip")
print("\n📥 Download from Google Drive:")
print(f"   {INDICBERT_DRIVE_OUTPUT}/indicbert/final/")
print("\n📋 To use locally:")
print("   1. Download the zip")
print("   2. Extract to: models/checkpoints/indicbert/")
print("   3. Run evaluation: python scripts/evaluate_model.py --model indicbert")
print("\n📁 Drive Paths Summary:")
print(f"   MuRIL:     /content/drive/MyDrive/telugu-qa-checkpoints/muril/")
print(f"   IndicBERT: /content/drive/MyDrive/telugu-qa-BERT-checkpoints/indicbert/")

---

## 📋 Training Summary

**Models Trained:**

| Model | Parameters | Location | Expected Performance |
|-------|------------|----------|---------------------|
| MuRIL | 237M | `checkpoints/muril/checkpoint-9000` | ~73% EM, ~84% F1 |
| IndicBERT | 33M | `checkpoints/indicbert/final` | ~65-70% EM, ~78-82% F1 |

**Next Steps:**
1. Download both model zips
2. Extract to `models/checkpoints/` locally
3. Run evaluations:
   ```bash
   python scripts/evaluate_model.py --model muril
   python scripts/evaluate_model.py --model indicbert
   ```
4. Start Streamlit app to see comparative results in sidebar!